# Prompt-injection detection baselines + second-model scan

This notebook adds the two experiments that reviewers asked for:

**A. Detector comparison** — our L1 marker detector vs **three SOTA prompt-injection
detectors** (ProtectAI DeBERTa, FMOPS DistilBERT, an LLM-as-judge), on our
English+Uzbek corpus **and** the public `deepset/prompt-injections` benchmark.

**B. Second target model** — re-run the attack-success scan on a second LLM so the
script effect is not tied to one model.

**Before running:** Settings → Accelerator → **GPU T4 x2**, and Settings →
**Internet: On**.


## 0. Setup: install packages and clone the code

In [ ]:
!pip -q install "transformers>=4.44" "datasets>=2.20" accelerate sentencepiece bitsandbytes scikit-learn 2>/dev/null
import os
if not os.path.exists("Promt-injection"):
    !git clone -q https://github.com/Mansurbekkk/Promt-injection.git
%cd Promt-injection
!git pull -q
print("files:", os.listdir("."))

## 1. (Optional) Hugging Face login — only for gated models

`Prompt-Guard-86M` (SOTA detector) and `Llama-3.1-8B` (second model) are gated on
Hugging Face. To use them: accept their licence on their HF model page, create a
token at huggingface.co/settings/tokens, and paste it below. **You can skip this**
— the notebook falls back to non-gated models that need no token.

In [ ]:
from huggingface_hub import login
HF_TOKEN = ""   # <-- paste your token here to enable gated models, or leave empty
if HF_TOKEN:
    login(HF_TOKEN)
    print("logged in to Hugging Face")
else:
    print("no token: gated models (Prompt-Guard, Llama) will be skipped / replaced")

## 2. Experiment A — detector comparison

Runs our L1 vs DeBERTa, DistilBERT and the LLM-judge on our corpus (EN + Uzbek
Latin/Cyrillic/mixed) and on the public benchmark. Add `--with-prompt-guard` if
you logged in above. ~15–25 min on a T4.

In [ ]:
PG = "--with-prompt-guard" if HF_TOKEN else ""
!python detect_baselines.py --out detect_results.json {PG}
import json; print(json.dumps(json.load(open("detect_results.json"))["corpus"]["results"]["our_l1"], indent=2))

## 3. Experiment B — second target model

Default is **Llama-3.1-8B-Instruct** (gated: needs the HF token above). If you did
not log in, set `MODEL2` to the non-gated alternative on the next line
(`Qwen/Qwen2.5-14B-Instruct`, a larger model of a different size, or
`microsoft/Phi-3.5-mini-instruct`, a different family). ~1–3 h on a T4.

In [ ]:
# pick the second model:
if HF_TOKEN:
    MODEL2 = "meta-llama/Llama-3.1-8B-Instruct"
else:
    MODEL2 = "microsoft/Phi-3.5-mini-instruct"   # non-gated fallback (different family)
print("second model:", MODEL2)
!python run.py --model {MODEL2} --load-in-4bit --out runs_model2.jsonl
!python analyze.py runs_model2.jsonl

## 4. Collect results to download
All outputs are written to the working directory; download them from the right-hand **Output** panel.

In [ ]:
import json, shutil, os
os.makedirs("/kaggle/working/results_new", exist_ok=True)
for f in ["detect_results.json", "runs_model2.jsonl"]:
    if os.path.exists(f):
        shutil.copy(f, "/kaggle/working/results_new/")
# quick summary of the detector comparison
d = json.load(open("detect_results.json"))
print("=== Detector comparison (corpus) — F1 by detector x script ===")
hdr = ["detector"] + d["scripts"]
print("{:12s}".format(hdr[0]), "  ".join(f"{s:>9s}" for s in d["scripts"]))
for name, per in d["corpus"]["results"].items():
    print("{:12s}".format(name), "  ".join(f"{per[s]['f1']:>9.3f}" for s in d["scripts"]))
print("\nFiles in /kaggle/working/results_new:", os.listdir("/kaggle/working/results_new"))